## APT
Advanced Package Tool is the package manager on Debian based Linux distributions. It can be used to install and remove debian packages (.deb).

## Installing Package
To download and install package:
```sh
$ sudo apt install git
```
To automatically accept add `-y` flag:
```sh
$ sudo apt install -y git
```

So how is the package located? When we run `apt update`, a number of files are consulted. These are present in `/etc/apt/sources.list` file and `/etc/apt/sources.list.d` directory. An example file `/etc/apt/sources.list.d/ubuntu.sources` has contents:
```
Types: deb
URIs: http://archive.ubuntu.com/ubuntu/
Suites: resolute resolute-updates resolute-backports
Components: main restricted universe multiverse
Signed-By: /usr/share/keyrings/ubuntu-archive-keyring.gpg

Types: deb
URIs: http://security.ubuntu.com/ubuntu/
Suites: resolute-security
Components: main restricted universe multiverse
Signed-By: /usr/share/keyrings/ubuntu-archive-keyring.gpg
```

Another one `/etc/apt/sources.list.d/steam-stable.list` has contents:
```
deb [arch=amd64,i386 signed-by=/usr/share/keyrings/steam.gpg] https://repo.steampowered.com/steam/ stable steam
deb-src [arch=amd64,i386 signed-by=/usr/share/keyrings/steam.gpg] https://repo.steampowered.com/steam/ stable steam
```

The content of these files essentially represent various package repositories. *Suite* refers to release channel (resolute, stable, etc). *Component* is the category of software:
- main: officially supported open-source
- restricted: proprietary drivers
- universe: community-maintained open-source
- multiverse: restricted by copyright/legal issues
Both of the above form the actual path APT fetches like `http://archive.ubuntu.com/ubuntu/dists/jammy/main/binary-amd64/Packages` or `https://repo.steampowered.com/steam/dists/stable/steam/binary-amd64/Packages`.

Other repositories can be added. Either manually:
```sh
# 1. Add Docker's GPG key
curl -fsSL https://download.docker.com/linux/ubuntu/gpg | \
sudo gpg --dearmor -o /usr/share/keyrings/docker.gpg

# 2. Add the repo source
echo "deb [arch=amd64 signed-by=/usr/share/keyrings/docker.gpg] https://download.docker.com/linux/ubuntu jammy stable" | \
sudo tee /etc/apt/sources.list.d/docker.list
```

Or through PPA route:
```sh
# Add the PPA, this automatically adds the source AND imports the GPG key
sudo add-apt-repository ppa:deadsnakes/ppa
```

Next, all the repository URLs are downloaded (and verified using the GPG key). The downloaded data is stored in `/var/lib/apt/lists`. One such file could be `repo.steampowered.com_steam_dists_stable_steam_binary-amd64_Packages`. Looking into its content we would find entries like:
```
Package: steam-launcher
Source: steam
Version: 1:1.0.0.87
Architecture: amd64
Maintainer: Valve Corporation <linux@steampowered.com>
Installed-Size: 20091
Pre-Depends: dpkg (>= 1.17.0)
Depends: apt (>= 1.1), apt (>= 1.6) | apt-transport-https, ca-certificates, coreutils (>= 8.23-1~), curl, default-dbus-session-bus | dbus-session-bus | dbus-x11, file, libc6 (>= 2.15), libnss3 (>= 2:3.26), lsof, pkexec | policykit-1, python3 (>= 3.4), python3-apt, xdg-user-dirs, xterm | gnome-terminal | konsole, xz-utils, zenity
Recommends: steam-libs-amd64, steam-libs-i386, sudo, xdg-desktop-portal, xdg-desktop-portal-gtk | xdg-desktop-portal-backend, xdg-utils | steamos-base-files
Conflicts: steam-devices, steam-installer
Breaks: steam64
Replaces: steam, steam-devices, steam-installer, steam64
Provides: steam, steam-devices, steam-installer
Multi-Arch: foreign
Homepage: http://www.steampowered.com/
Priority: optional
Section: games
Filename: pool/steam/s/steam/steam-launcher_1.0.0.87_amd64.deb
Size: 20379390
SHA256: 765aba9a0ed339a50226ceb614fcc9879a991ba184098bc8de920efb12c714a4
SHA1: 7b06640a1130dd3b364b4df00216905d54e4208b
MD5sum: 380048701b195262f5c9f888a45dfc49
Description: Launcher for the Steam software distribution service
 Steam is a software distribution service with an online store, automated
 installation, automatic updates, achievements, SteamCloud synchronized
 savegame and screenshot functionality, and many social features.
```

Each package entry has:
- Package: entry's actual package name
- Version
- Depends: hard requirements which must be installed
- Conflicts: can't be installed alongside these packages
- Filename: of the `.deb` file

Instead of specifying a package name, we can directly point to a `.deb` file as well:
```sh
$ sudo apt install ./package_file.deb
```

### Conflicting Dependencies
Lets say package `A` depends upon package `X`. Package `B` also depends upon `X`. What could be the scenarios?
1. `A` depends on `X` >= 2.0.  
   `B` depends on 3.0 >= `X` >= 1.5  
    Resolvable ✅

1. `A` depends on `X` >= 2.0.  
   `B` depends on `X` <= 1.5  
    Conflict ❌

In case of conflict APT tries several options including removing one package to install another or alternative package. If all fails, installation fails.

To overcome this sort of problem, package maintainers may also name package in a way that includes version number. Like:
```
libssl1.1
libssl3
```

## `.deb` Internals
A `.deb` file is essentially an archive, on extracting it, we would see the following:
```sh
$ ar -tv Downloads/git.deb
rw-r--r-- 0/0      4 Aug  2 20:09 2026 debian-binary
rw-r--r-- 0/0    896 Aug  2 20:09 2026 control.tar.xz
rw-r--r-- 0/0 1150860 Aug  2 20:09 2026 data.tar.xz
```

`debian-binary` file just contains the format version number.

`control.tar.xz` contains metadata and scrupts about the package. Some files that may be present are:
- control: package entry as we saw earlier
- md5sums: checksums of every file that WILL be installed
- preinst: script run BEFORE unpacking files
- postinst: script run AFTER unpacking files (e.g., register services, run ldconfig)
- prerm: script run BEFORE removal
- postrm: script run AFTER removal
- conffiles: list of config files that shouldn't be overwritten on upgrade
- templates: debconf prompts, if package asks interactive questions

As an example VSCode `postinst` script roughly looks like:
```sh
# System Integration (First ~15 lines)
rm -f /usr/bin/code
ln -s /usr/share/code/bin/code /usr/bin/code

# Registers VS Code as a candidate for the system's editor alternative
if hash update-desktop-database 2>/dev/null; then
	update-desktop-database
fi

# Refreshes the desktop application database so VS Code shows up in app launchers
if hash update-mime-database 2>/dev/null; then
	update-mime-database /usr/share/mime
fi

# Other
```

The `prerm` and `postrm` scripts revert the above changes.

`data.tar.xz` contains the actual application data that gets copied to the filesystem. Exactly as they appear in disk. If we check the contents of data after extracting we see:
```sh
$ tree Downloads/code_1.139.1-1790309529_amd64/data
Downloads/code_1.139.1-1790309529_amd64/data
└── usr
    └── share
        ├── appdata
        │   └── com.microsoft.VSCode.appdata.xml
        ├── applications
        │   ├── com.microsoft.VSCode.desktop
        │   └── com.microsoft.VSCode.UrlHandler.desktop
        ├── bash-completion
        │   └── completions
        │       └── code
        ├── code
        │   ├── bin
        │   │   ├── code
        │   │   └── code-tunnel
        |   ... many more files and directories
        ├── mime
        │   └── packages
        │       └── code-workspace.xml
        ├── pixmaps
        │   └── vscode.png
        └── zsh
            └── vendor-completions
                └── _code
```

This shows that most of the application files will be installed in `/usr/share/code` directory.

## `dpkg`
Debian Package is the low-level package manager that actually installs, removes, and manages `.deb` files on the system. It's the foundation that APT is built on top of. When we call `apt install steam`:
1. APT resolves the dependencies
2. Downloads `.deb` files from the repo
3. Calls `dpkg` internally to install each `.deb` file
4. `dpkg` unpacks `data.tar.gz` and runs scripts inside `control.tar.gz`.

During installation, it also records the package ownership of each file. This information is present in `/var/lib/dpkg/info/` and `/var/lib/dpkg/status`. During removal these locations are consulted. The control scripts are copied here as well.

## Removing Packages
To remove a package, run:
```sh
$ sudo apt remove git
```

This removes all package files except for configuration files. The `conffiles` file inside `control.tar.gz` lists all configuration files (typically placed in `/etc/`).

To remove package + configuration files:
```sh
$ sudo apt purge git
```

### Removing unused dependencies
When we install a package, its dependencies listed under `Depends` line are also installed. When the package is removed, those dependencies (if no other package depends upon it) are orphaned.

Running:
```sh
# Add --purge flag to purge dependencies
$ sudo apt autoremove
```
cleans such dependencies.

## Listing Packages
To see all packages from all configured repositories, run:
```sh
$ sudo apt list
```

To list only installed package:
```sh
$ sudo apt list --installed
```

We can also use `dpkg`:
```sh
$ dpkg -l
┌── Desired=Unknown/Install/Remove/Purge/Hold
│┌─ Status=Not/Inst/Conf-files/Unpacked/halF-conf/Half-inst/trig-aWait/Trig-pend
││┌ Err?=(none)/Reinst-required (Status,Err: uppercase=bad)
│││ Name               Version              Architecture Description                                      
├┼┼─══════════════════─════════════════════─════════════─═══════════════════════════════════════════════>
ii  3cpio              0.14.0-1ubuntu1      amd64        Manage initrd cpio archives
ii  accountsservice    23.13.9-8ubuntu5.2   amd64        query and manipulate user account information
ii  acl                2.3.2-2              amd64        access control list - utilities
ii  adduser            3.153ubuntu1         all          add and remove users and group


## System Wide vs Per User Installation
APT doesn't have concept of per user installation (use of `sudo` hints at this). Even though the binary/program files are shared, individual users still get their own:
- User configurations are often placed in `~/.config/`, `~/.<app>rc`.
- User data/cache (like extensions) are placed in `~/.local/share/`, `~/.cache/`.
<div style="display:inline-block">

|                      | `/etc/*`                                                    | `~/.config/*` (or `~/.apprc`)                               |
|----------------------|-------------------------------------------------------------|-------------------------------------------------------------|
| **Managed by**       | dpkg (conffiles mechanism)                                  | Nobody, pure application logic                              |
| **Scope**            | System-wide default, same for every user                    | Per-user                                                    |
| **Touched during**   | `apt install/upgrade/remove/purge`                          | Never touched by `dpkg` at all                              |
| **Upgrade behavior** | `dpkg` may prompt to merge/keep/overwrite (conffile prompt) | App itself decides how to handle version changes, if at all |
</div>

`conffile` can never contain `~`, so no configuration file is created (and thus managed by `dpkg`) during installation. It is upto the application how to handle configuration in `/etc/` vs inside the user's home directory. Usually the latter is given preference. Example:
```
/etc/gitconfig            # system-wide, --system scope
~/.gitconfig              # per-user, --global scope
./.git/config             # per-repo, --local scope (highest priority)
```

Another example:
```
/etc/vim/vimrc            # ships with vim package, is a conffile, applies to every user by default
~/.vimrc                  # if it exists, user's vim reads this AFTER /etc/vim/vimrc
```